# Piecewise McCormick Gap Reduction (Table 4)

This notebook reproduces the piecewise McCormick gap reduction numbers for Table 4 of the
manuscript. For each univariate function we compare the gap `cc - cv` of the standard envelope
over the whole interval with the gap of the piecewise envelope on `k = 8` uniform partitions
(Castro 2015), and report the percentage reduction.

The envelopes are the ones discopt's default relaxation engine emits. For a function of fixed
curvature on an interval, the tight convex/concave envelope pair is the function itself on one
side and the secant through the endpoints on the other: for convex `exp` and `x^2`,
`cv = f(x)` and `cc = secant`; for concave `log`, `cv = secant` and `cc = f(x)`. The piecewise
envelope is the same pair on the partition cell that contains `x`. The function values and the
curvature verdicts are taken from the engine's own table (`uniform_relax._UNIVARIATE_FN`, the
table the partition path `_PIECEWISE_UNIVARIATE_FN` is built from), so the numbers below are
computed from the same `f` and curvature the solver uses. The earlier JAX primitives
(`discopt._relax.mccormick`, `discopt._relax.piecewise_mccormick`) were retired with the
McCormick-NLP route in #1873.

Because the envelopes are closed-form, the mean gaps also have exact values, and the last cell
checks the sampled means against them.

In [1]:
import numpy as np
from discopt._relax.uniform_relax import _UNIVARIATE_FN

N_POINTS = 10_000
K = 8  # number of piecewise partitions


def envelope(fname, x, lb, ub):
    """(cv, cc) of the fixed-curvature function ``fname`` over [lb, ub] at x."""
    f, _fp, curv_fn, dom_ok = _UNIVARIATE_FN[fname]
    assert dom_ok(lb), (fname, lb)
    curvature = curv_fn(lb, ub)
    secant = f(lb) + (f(ub) - f(lb)) / (ub - lb) * (x - lb)
    if curvature == "convex":
        return f(x), secant
    if curvature == "concave":
        return secant, f(x)
    raise ValueError(f"{fname} has no single curvature on [{lb}, {ub}]: {curvature}")


def piecewise_envelope(fname, x, lb, ub, k=K):
    """(cv, cc) of the envelope on the uniform partition cell containing x."""
    edges = np.linspace(lb, ub, k + 1)
    cell = np.clip(np.searchsorted(edges, x, side="right") - 1, 0, k - 1)
    cv = np.empty_like(x)
    cc = np.empty_like(x)
    for j in range(k):
        mask = cell == j
        cv[mask], cc[mask] = envelope(fname, x[mask], edges[j], edges[j + 1])
    return cv, cc


def compute_gap_reduction(fname, lb, ub, rng):
    """(std_gap, pw_gap, reduction_pct) with gaps = mean(cc - cv) over uniform samples."""
    x = rng.uniform(lb, ub, N_POINTS)
    std_cv, std_cc = envelope(fname, x, lb, ub)
    pw_cv, pw_cc = piecewise_envelope(fname, x, lb, ub)
    assert np.all(std_cv <= std_cc + 1e-12) and np.all(pw_cv <= pw_cc + 1e-12)
    # The piecewise envelope must be nested inside the standard one.
    assert np.all(pw_cv >= std_cv - 1e-12) and np.all(pw_cc <= std_cc + 1e-12)
    std_gap = float(np.mean(std_cc - std_cv))
    pw_gap = float(np.mean(pw_cc - pw_cv))
    return std_gap, pw_gap, (1.0 - pw_gap / max(std_gap, 1e-15)) * 100.0

In [2]:
# (name, intrinsic in _UNIVARIATE_FN, interval)
test_cases = [
    ("exp(x)", "exp", -2.0, 3.0),
    ("log(x)", "log", 0.5, 8.0),
    ("x^2", "square", -3.0, 4.0),
]

if "square" not in _UNIVARIATE_FN:  # x^2 is a power node in the engine, not an intrinsic
    _UNIVARIATE_FN = dict(_UNIVARIATE_FN)
    _UNIVARIATE_FN["square"] = (np.square, lambda t: 2 * t, lambda lo, hi: "convex", lambda lo: True)

results = []
for i, (name, fname, lb, ub) in enumerate(test_cases):
    rng = np.random.default_rng(42 + i)
    std_gap, pw_gap, reduction = compute_gap_reduction(fname, lb, ub, rng)
    results.append(
        {
            "name": name,
            "fname": fname,
            "interval": f"[{lb:g}, {ub:g}]",
            "lb": lb,
            "ub": ub,
            "std_gap": std_gap,
            "pw_gap": pw_gap,
            "reduction": reduction,
        }
    )

In [3]:
# Print Table 4
print("Table 4: Piecewise McCormick Gap Reduction (k=8 uniform partitions)")
print("=" * 75)
print(f"{'Function':<12} {'Interval':<12} {'Std Gap':>12} {'PW Gap (k=8)':>14} {'Reduction':>12}")
print("-" * 75)
for r in results:
    print(
        f"{r['name']:<12} {r['interval']:<12}"
        f" {r['std_gap']:>12.6f} {r['pw_gap']:>14.6f} {r['reduction']:>11.1f}%"
    )
print("=" * 75)
print(f"\nN = {N_POINTS} random sample points per function.")
print("Gap = mean(cc - cv) over sample points.")
print("Reduction = (1 - pw_gap / std_gap) * 100%.")

Table 4: Piecewise McCormick Gap Reduction (k=8 uniform partitions)
Function     Interval          Std Gap   PW Gap (k=8)    Reduction
---------------------------------------------------------------------------
exp(x)       [-2, 3]          6.109870       0.126597        97.9%
log(x)       [0.5, 8]         0.568153       0.017075        97.0%
x^2          [-3, 4]          8.137854       0.128261        98.4%

N = 10000 random sample points per function.
Gap = mean(cc - cv) over sample points.
Reduction = (1 - pw_gap / std_gap) * 100%.


## Exact mean gaps

For a fixed-curvature `f` on `[a, b]` the mean envelope gap is
`(f(a) + f(b)) / 2 - (F(b) - F(a)) / (b - a)` (the secant's mean minus `f`'s mean, up to sign),
with `F` an antiderivative. The piecewise mean is the average of that quantity over the `K`
cells. For `x^2` this is `(b - a)^2 / 6` on the whole interval and `(b - a)^2 / (6 K^2)`
piecewise, a reduction of exactly `1 - 1/K^2 = 98.4%`.

In [4]:
ANTIDERIVATIVE = {
    "exp": np.exp,
    "log": lambda t: t * np.log(t) - t,
    "square": lambda t: t**3 / 3,
}


def exact_gap(fname, a, b):
    f = _UNIVARIATE_FN[fname][0]
    F = ANTIDERIVATIVE[fname]
    return abs((f(a) + f(b)) / 2 - (F(b) - F(a)) / (b - a))


n_checked = 0
print(f"{'Function':<10} {'exact std':>12} {'exact PW':>12} {'exact red.':>11}")
for r in results:
    edges = np.linspace(r["lb"], r["ub"], K + 1)
    std_exact = exact_gap(r["fname"], r["lb"], r["ub"])
    pw_exact = np.mean([exact_gap(r["fname"], a, b) for a, b in zip(edges[:-1], edges[1:])])
    print(
        f"{r['name']:<10} {std_exact:>12.6f} {pw_exact:>12.6f}"
        f" {(1 - pw_exact / std_exact) * 100:>10.1f}%"
    )
    # Sampling error of a 10k-point mean is well under 3% here.
    assert abs(r["std_gap"] - std_exact) <= 0.03 * std_exact
    assert abs(r["pw_gap"] - pw_exact) <= 0.03 * pw_exact
    n_checked += 2
assert n_checked == 2 * len(results)
print(f"\n{n_checked} sampled means agree with the exact values to within 3%.")

Function      exact std     exact PW  exact red.
exp(x)         6.120396     0.129046       97.9%
log(x)         0.571134     0.016893       97.0%
x^2            8.166667     0.127604       98.4%

6 sampled means agree with the exact values to within 3%.
